In [1]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch_geometric
import rdkit

from rdkit import Chem
from rdkit.Chem import Descriptors

from torch_geometric.data import Data

In [2]:
seed = 42
random.seed(seed)
np.random.seed(seed)
os.environ["PYTHONHASHSEED"] = str(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=== Versions: ===")
print("rdkit:\t", rdkit.__version__)
print("torch:\t", torch.__version__)
print("PyG:\t", torch_geometric.__version__)
print("device:\t", DEVICE)
if torch.cuda.is_available():
    print("CUDA:\t", torch.version.cuda)


=== Versions: ===
rdkit:	 2025.09.6
torch:	 2.10.0+cu128
PyG:	 2.7.0
device:	 cuda
CUDA:	 12.8


In [3]:
# BLOCK: File paths

DATA_DIR = '../data/'

DATA_PATH = DATA_DIR + 'Active_Database.csv'
ACCEPTOR_EMBEDDINGS = DATA_DIR + 'acceptor_embeddings_pca250.csv'
DONOR_EMBEDDINGS = DATA_DIR + 'acceptor_embeddings_pca250.csv'


df = pd.read_csv(DATA_PATH, encoding="latin1")
donor_emb = pd.read_csv(DONOR_EMBEDDINGS)
acceptor_emb = pd.read_csv(ACCEPTOR_EMBEDDINGS)

In [4]:
# BLOCK: Add ChemBERTa embeddings, clean dataframe

# Add new columns for ChemBERTa embeddings
df["D_embedding"] = pd.Series()
df["A_embedding"] = pd.Series()

for index, row in df.iterrows():
    df.at[index, "D_embedding"] = donor_emb.loc[index, :].values.flatten().tolist()
    df.at[index, "A_embedding"] = acceptor_emb.loc[index, :].values.flatten().tolist()

required = ["Donor SMILES", "Acceptor SMILES", "HOMO_D", "LUMO_D", "HOMO_A", "LUMO_A", "PCE"]

# Reduce to only checked values
df = df[df["Checked"].astype(str).str.upper() == "YES"].copy()


required = ["Donor SMILES", "Acceptor SMILES", "HOMO_D", "LUMO_D", "HOMO_A", "LUMO_A", "PCE"]
df = df.dropna(subset=required)
    
df = df.drop_duplicates(subset=["Donor SMILES", "Acceptor SMILES"]).reset_index(drop=True)


df["HOMO_D"] = pd.to_numeric(df["HOMO_D"], errors="coerce")
df["LUMO_D"] = pd.to_numeric(df["LUMO_D"], errors="coerce")
df["HOMO_A"] = pd.to_numeric(df["HOMO_A"], errors="coerce")
df["LUMO_A"] = pd.to_numeric(df["LUMO_A"], errors="coerce")
df["PCE"]    = pd.to_numeric(df["PCE"], errors="coerce")
    
df["dHOMO"] = df["HOMO_D"] - df["HOMO_A"]
df["dLUMO"] = df["LUMO_A"] - df["LUMO_D"]

df = df.dropna(subset=["dHOMO", "dLUMO", "PCE"]).reset_index(drop=True)
print("After basic clean:", df.shape)

After basic clean: (1776, 22)


In [5]:
# BLOCK: Add rdkit molecules

df["D_mol"] = df["Donor SMILES"].apply(Chem.MolFromSmiles)
df["A_mol"] = df["Acceptor SMILES"].apply(Chem.MolFromSmiles)
    
before = len(df)
df = df.dropna(subset=["D_mol", "A_mol"]).reset_index(drop=True)
print("Dropped invalid SMILES:", before - len(df))
    
df["D_MolWt"] = df["D_mol"].apply(Descriptors.MolWt)
df["A_MolWt"] = df["A_mol"].apply(Descriptors.MolWt)
    
df = df.dropna(subset=["D_MolWt", "A_MolWt"]).reset_index(drop=True)
print("Final clean:", df.shape)
    
df.head()

Dropped invalid SMILES: 1
Final clean: (1775, 26)


[15:06:10] SMILES Parse Error: syntax error while parsing: 
[15:06:10] SMILES Parse Error: check for mistakes around position 1:
[15:06:10] 
[15:06:10] ^
[15:06:10] SMILES Parse Error: Failed parsing SMILES ' ' for input: ' '


,Checked,DOI,Donor,Acceptor,Donor SMILES,Acceptor SMILES,Voc,Jsc,FF,PCE,...,Ehl_D,Eg_D,D_embedding,A_embedding,dHOMO,dLUMO,D_mol,A_mol,D_MolWt,A_MolWt
0,YES,10.1002/aenm.201703291,PBFSF,N2200,CCCCCCC(CCCC)CC1=CC=C(C2=C3SC=CC3=C(C3=CC=C(CC...,CCCCCCCCCCC(CCCCCCCC)CN1C(=O)C2=C3C(=CC(C4=CC=...,0.83,11.95,0.61,6.00,...,1.55,NaN,"[5.505321, -9.4000025, -1.3747191, -4.2522874,...","[5.505321, -9.4000025, -1.3747191, -4.2522874,...",0.44,-0.30,<rdkit.Chem.rdchem.Mol object at 0x7fd0151cc900>,<rdkit.Chem.rdchem.Mol object at 0x7fd012a21b60>,1412.272,991.546
1,YES,10.1002/aenm.201703291,PBBSB,N2200,CCCCCCC(CCCC)CC1=CC=C(C2=C3SC=CC3=C(C3=CC=C(CC...,CCCCCCCCCCC(CCCCCCCC)CN1C(=O)C2=C3C(=CC(C4=CC=...,0.73,7.81,0.52,3.00,...,1.56,NaN,"[5.5052915, -9.400042, -1.3746015, -4.252259, ...","[5.5052915, -9.400042, -1.3746015, -4.252259, ...",0.52,-0.39,<rdkit.Chem.rdchem.Mol object at 0x7fd0151ccb30>,<rdkit.Chem.rdchem.Mol object at 0x7fd012a21bd0>,1376.292,991.546
2,YES,10.1002/aenm.201801209,PTB7-Th,IOTIC-2F,CCCCC(CC)COC(=O)c1sc2c(-c3cc4c(-c5ccc(CC(CC)CC...,CCCCCCC1=CC=C(C2(C3=CC=C(CCCCCC)C=C3)C3=C(C=C4...,0.817,21.90,0.65,12.10,...,1.61,1.58,"[16.245647, 4.3145394, -2.5791366, 3.0883837, ...","[16.245647, 4.3145394, -2.5791366, 3.0883837, ...",0.13,-0.46,<rdkit.Chem.rdchem.Mol object at 0x7fd0151cc510>,<rdkit.Chem.rdchem.Mol object at 0x7fd012a21c40>,919.463,1772.470
3,YES,10.1002/aenm.201901024,PTB7-Th,3TT-CIC,CCCCC(CC)COC(=O)c1sc2c(-c3cc4c(-c5ccc(CC(CC)CC...,CCCCCCC1=CC=C(C2(C3=CC=C(CCCCCC)C=C3)C3=C(SC4=...,0.65,26.67,0.69,11.96,...,1.61,1.58,"[18.649843, 5.1467776, -2.899041, 2.8863094, 2...","[18.649843, 5.1467776, -2.899041, 2.8863094, 2...",0.03,-0.35,<rdkit.Chem.rdchem.Mol object at 0x7fd0151ccc10>,<rdkit.Chem.rdchem.Mol object at 0x7fd012a21cb0>,919.463,1627.882
4,YES,10.1002/adma.201706816,PTB7-Th,6TBA,CCCCC(CC)COC(=O)c1sc2c(-c3cc4c(-c5ccc(CC(CC)CC...,CCCCCCC1=CC=C(C2(C3=CC=C(CCCCCC)C=C3)C3=C(SC4=...,0.98,15.20,0.68,10.13,...,1.61,1.58,"[19.651604, 3.9132411, -4.1987934, 2.8861928, ...","[19.651604, 3.9132411, -4.1987934, 2.8861928, ...",0.03,-0.18,<rdkit.Chem.rdchem.Mol object at 0x7fd0151ccc80>,<rdkit.Chem.rdchem.Mol object at 0x7fd012a21d20>,919.463,1502.242


In [6]:
HYB_MAP = {
    Chem.rdchem.HybridizationType.SP: 0, 
    Chem.rdchem.HybridizationType.SP2: 1,
    Chem.rdchem.HybridizationType.SP3: 2,
    Chem.rdchem.HybridizationType.SP3D: 3,
    Chem.rdchem.HybridizationType.SP3D2: 4,
}
BOND_MAP = {
    Chem.rdchem.BondType.SINGLE: 0,
    Chem.rdchem.BondType.DOUBLE: 1,
    Chem.rdchem.BondType.TRIPLE: 2,
    Chem.rdchem.BondType.AROMATIC: 3,
}

def one_hot(idx: int, size: int):
       v = [0.0] * size
       if 0 <= idx < size:
           v[idx] = 1.0
       return v

def row_to_graph(df_row: pd.Series):
    """
    @df_row: []
    """
    if ["D_mol", "D_embedding"] in df_row.index.array:
        mol: Chem.Mol = df_row.get("D_mol")
        emb = df_row.get("D_embedding")
    
    elif ["A_mol", "A_embedding"] in df_row.index.array:
        mol: Chem.Mol = df_row.get("A_mol")
        emb = df_row.get("A_embedding")
    else:
        raise ValueError("Required columns not found")
    
    node_feats = []
    for atom in mol.GetAtoms():
        node_feats.append([
            float(atom.GetAtomicNum()),
            float(atom.GetDegree()),
            float(atom.GetFormalCharge()),
            float(HYB_MAP.get(atom.GetHybridization(), -1)),
            float(int(atom.GetIsAromatic())),
            float(atom.GetTotalNumHs(includeNeighbors=True)),
            float(int(atom.IsInRing()))
        ])
    x = torch.tensor(node_feats, dtype=torch.float)
    
    edge_index = []
    edge_attr  = []
    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()
    
        btype = BOND_MAP.get(bond.GetBondType(), -1)
        b_oh = one_hot(btype, 4)
    
        e = b_oh + [float(int(bond.GetIsConjugated())), float(int(bond.IsInRing()))]
    
        edge_index.append([i, j]); edge_attr.append(e)
        edge_index.append([j, i]); edge_attr.append(e)
    
    if len(edge_index) == 0:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_attr  = torch.zeros((0, 6), dtype=torch.float)
    else:
        edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
        edge_attr  = torch.tensor(edge_attr, dtype=torch.float)

    # Graph features (embeddings)
    graph_attr = torch.tensor([emb])
    
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, graph_attr=graph_attr)


df["D_graph"] = df[["D_mol", "D_embedding"]].apply(row_to_graph, axis=1)
df["A_graph"] = df[["A_mol", "A_embedding"]].apply(row_to_graph, axis=1)

before = len(df)
df = df.dropna(subset=["D_graph", "A_graph"]).reset_index(drop=True)
print("Dropped graph failures:", before - len(df))
print("Graph-ready:", df.shape)

Dropped graph failures: 0
Graph-ready: (1775, 28)


In [7]:
torch.save(df, DATA_DIR + "graphs_cached.pt")